# Logistics Parcel Volume Forecasting — Baseline & Linear Regression

This notebook develops the first version of a parcel-volume forecasting model for a **simulated logistics operation**.

The workflow is intentionally business-focused:

1. understand the dataset,
2. explore operational patterns,
3. establish a simple baseline,
4. engineer historical/calendar features,
5. use a time-based train/test split,
6. train Linear Regression,
7. compare it fairly with the baseline.

> The dataset is synthetic. No confidential or real DHL operational data is used.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATA_PATH = Path("../data/dhl_style_parcel_volume.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df.head()

## 1. Initial data checks

Before modelling, inspect the shape, columns, data types, missing values and summary statistics.

In [ ]:
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("\nMissing values:")
print(df.isna().sum())

In [ ]:
df.describe(include="all")

## 2. Exploratory data analysis

The first questions are operational rather than purely technical:

- Which depot processes the most parcels on average?
- Which day of the week is busiest?
- How does Christmas peak demand compare with normal periods?
- Are weekends quieter than weekdays?
- Is there evidence of a longer-term volume trend?

In [ ]:
# Average parcel volume by depot
avg_by_depot = (
    df.groupby("depot")["parcel_volume"]
    .mean()
    .sort_values(ascending=False)
)
avg_by_depot

In [ ]:
# Average parcel volume by day of week
df["day_of_week"] = df["date"].dt.day_name()

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

avg_by_day = (
    df.groupby("day_of_week")["parcel_volume"]
    .mean()
    .reindex(day_order)
)

avg_by_day

In [ ]:
avg_by_day.plot(kind="bar", figsize=(9, 4))
plt.title("Average Parcel Volume by Day of Week")
plt.ylabel("Average parcels")
plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Christmas peak vs normal periods
christmas_avg = df.loc[
    df["is_christmas_peak"] == 1, "parcel_volume"
].mean()

normal_avg = df.loc[
    df["is_christmas_peak"] == 0, "parcel_volume"
].mean()

christmas_increase_pct = ((christmas_avg - normal_avg) / normal_avg) * 100

print(f"Christmas peak average: {christmas_avg:,.0f}")
print(f"Normal-period average: {normal_avg:,.0f}")
print(f"Increase: {christmas_increase_pct:.1f}%")

In [ ]:
# Weekday vs weekend comparison
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])

df.groupby("is_weekend")["parcel_volume"].mean().rename(
    index={False: "Weekday", True: "Weekend"}
)

In [ ]:
# 30-day rolling average of total network volume
daily_total = df.groupby("date")["parcel_volume"].sum()
rolling_average = daily_total.rolling(30).mean()

rolling_average.plot(figsize=(14, 5))
plt.title("30-Day Rolling Average of Total Parcel Volume")
plt.ylabel("Parcels")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

## 3. Establish a forecasting baseline

A useful ML model should beat a simple alternative. The baseline used here assumes that a depot's parcel volume will be approximately the same as it was on the **same day one week earlier**.

That means the baseline prediction is simply `lag_7`.

In [ ]:
df = df.sort_values(["depot", "date"]).copy()
df["lag_7"] = df.groupby("depot")["parcel_volume"].shift(7)

df[["date", "depot", "parcel_volume", "lag_7"]].head(12)

## 4. Feature engineering

The first machine-learning model uses recent historical demand plus calendar information.

In [ ]:
df["lag_1"] = df.groupby("depot")["parcel_volume"].shift(1)
df["month"] = df["date"].dt.month
df["day_of_week_num"] = df["date"].dt.dayofweek

model_df = df.dropna(subset=["lag_1", "lag_7"]).copy()

features = [
    "lag_1",
    "lag_7",
    "day_of_week_num",
    "month",
    "is_bank_holiday",
    "is_black_friday_period",
    "is_christmas_peak",
]

X = model_df[features]
y = model_df["parcel_volume"]

X.head()

## 5. Time-based train/test split

Forecasting data should not be randomly shuffled because that can allow information from the future to leak into model development.

Here, observations before **1 January 2026** are used for training, while 2026 observations form the unseen test period.

In [ ]:
split_date = "2026-01-01"

train_df = model_df[model_df["date"] < split_date].copy()
test_df = model_df[model_df["date"] >= split_date].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

In [ ]:
X_train = train_df[features]
y_train = train_df["parcel_volume"]

X_test = test_df[features]
y_test = test_df["parcel_volume"]

## 6. Linear Regression

Linear Regression provides a simple machine-learning benchmark before moving to nonlinear models such as Random Forest or Gradient Boosting.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)
linear_mae = mean_absolute_error(y_test, linear_predictions)

print(f"Linear Regression MAE: {linear_mae:.2f}")

## 7. Fair baseline comparison

The original baseline should be evaluated on the **same test period** as the ML model. This makes the comparison apples-to-apples.

In [ ]:
baseline_test_mae = mean_absolute_error(
    test_df["parcel_volume"],
    test_df["lag_7"],
)

improvement_pct = (
    (baseline_test_mae - linear_mae) / baseline_test_mae
) * 100

results = pd.DataFrame(
    {
        "Model": ["7-day lag baseline", "Linear Regression"],
        "Test MAE": [baseline_test_mae, linear_mae],
    }
)

print(results.to_string(index=False))
print(f"\nImprovement vs baseline: {improvement_pct:.1f}%")

### Current result

Using the generated dataset and the time-based split above:

- **7-day baseline MAE:** approximately **1,182.84** parcels
- **Linear Regression MAE:** approximately **1,029.12** parcels
- **Improvement:** approximately **13%**

This means Linear Regression reduces the average absolute forecasting error by roughly 154 parcels per depot-day relative to the simple weekly baseline.

## Next steps

The next iteration will test nonlinear models and richer time-series features, including:

- Random Forest
- Gradient Boosting
- rolling means
- additional lag features
- model explainability
- predicted-vs-actual visualisations
